# K-Nearest Neighbors Training

This notebook loads the prepared scaled train and test parquet files from `../data/scaled/`, trains the `knn` model, saves the fitted model to `../model/`, and writes evaluation outputs to `../model/results/`.


## 1. Imports and Config

Set the shared file locations, output paths, and model-specific dependencies for this training run.


In [ ]:
from pathlib import Path

import polars as pl
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
from sklearn.neighbors import KNeighborsClassifier
import joblib

DATA_DIR = Path("..") / "data" / "scaled"
X_TRAIN_FILE = DATA_DIR / "X_train.parquet"
Y_TRAIN_FILE = DATA_DIR / "y_train.parquet"
X_TEST_FILE = DATA_DIR / "X_test.parquet"
Y_TEST_FILE = DATA_DIR / "y_test.parquet"
MODEL_DIR = Path("..") / "model"
RESULTS_DIR = MODEL_DIR / "results"
TARGET_COLUMN = "label"
RANDOM_SEED = 42
MODEL_NAME = "knn"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_FILE = MODEL_DIR / "knn_model.joblib"
RESULTS_FILE = RESULTS_DIR / "knn_results.csv"
METRICS_FILE = RESULTS_DIR / "knn_metrics.csv"


## 2. Load the Dataset

Load the prepared feature matrices and target vectors from the scaled data folder.


In [ ]:
for required_file in [X_TRAIN_FILE, Y_TRAIN_FILE, X_TEST_FILE, Y_TEST_FILE]:
    if not required_file.exists():
        raise FileNotFoundError(f"Required parquet file not found: {required_file}")

X_train = pl.read_parquet(X_TRAIN_FILE)
y_train_df = pl.read_parquet(Y_TRAIN_FILE)
X_test = pl.read_parquet(X_TEST_FILE)
y_test_df = pl.read_parquet(Y_TEST_FILE)

print(f"loaded X_train: {X_train.shape}")
print(f"loaded y_train: {y_train_df.shape}")
print(f"loaded X_test: {X_test.shape}")
print(f"loaded y_test: {y_test_df.shape}")


## 3. Separate Features and Target

Keep the feature matrices separate and extract the binary target from the `label` column in the target parquet files.


In [ ]:
y_train = y_train_df.get_column(TARGET_COLUMN).cast(pl.Int32)
y_test = y_test_df.get_column(TARGET_COLUMN).cast(pl.Int32)

print(f"feature columns: {len(X_train.columns)}")
print(f"target column: {TARGET_COLUMN}")


## 4. Validate the Prepared Features

Fail fast if the upstream split and scaling pipeline did not produce aligned numeric model inputs.


In [ ]:
for frame_name, frame in {"y_train": y_train_df, "y_test": y_test_df}.items():
    if TARGET_COLUMN not in frame.columns:
        raise ValueError(f"{frame_name}.parquet must include the target column: {TARGET_COLUMN}")
    if frame.width != 1:
        raise ValueError(f"{frame_name}.parquet should only contain the target column: {TARGET_COLUMN}")

if TARGET_COLUMN in X_train.columns or TARGET_COLUMN in X_test.columns:
    raise ValueError(f"X parquet files should only contain features, not the target column: {TARGET_COLUMN}")

if X_train.columns != X_test.columns:
    raise ValueError("X_train and X_test must contain the same feature columns in the same order.")

if X_train.height != y_train.len():
    raise ValueError("X_train and y_train must have the same number of rows.")

if X_test.height != y_test.len():
    raise ValueError("X_test and y_test must have the same number of rows.")

non_numeric_train = [column_name for column_name, dtype in X_train.schema.items() if not dtype.is_numeric()]
non_numeric_test = [column_name for column_name, dtype in X_test.schema.items() if not dtype.is_numeric()]
non_numeric_features = sorted(set(non_numeric_train + non_numeric_test))

if non_numeric_features:
    preview = ", ".join(non_numeric_features[:10])
    raise ValueError(
        "The split/feature-selection notebook must output model-ready numeric features. "
        f"Non-numeric features found: {preview}"
    )

print(f"validated {len(X_train.columns)} numeric features")


## 5. Train the Model

Train the K-nearest neighbors classifier on the prepared scaled feature matrix.


In [ ]:
X_train_np = X_train.to_numpy()
y_train_np = y_train.to_numpy()
X_test_np = X_test.to_numpy()
y_test_np = y_test.to_numpy()

model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train_np, y_train_np)

model


## 6. Evaluate and Save

Score the holdout set, save the trained model artifact, and export both metrics and row-level prediction results.


In [ ]:
probabilities = model.predict_proba(X_test_np)[:, 1]
predictions = (probabilities >= 0.5).astype(int)
actuals = y_test_np

metrics = {
    "model": MODEL_NAME,
    "accuracy": accuracy_score(actuals, predictions),
    "precision": precision_score(actuals, predictions, zero_division=0),
    "recall": recall_score(actuals, predictions, zero_division=0),
    "roc_auc": roc_auc_score(actuals, probabilities),
}

metrics_frame = pl.DataFrame([metrics])
results_frame = X_test.with_columns(
    [
        pl.Series("actual", actuals),
        pl.Series("predicted", predictions),
        pl.Series("probability", probabilities),
        pl.lit(MODEL_NAME).alias("model"),
    ]
)

joblib.dump(model, MODEL_FILE)
metrics_frame.write_csv(METRICS_FILE)
results_frame.write_csv(RESULTS_FILE)

print(f"model saved to: {MODEL_FILE}")
print(f"metrics saved to: {METRICS_FILE}")
print(f"results saved to: {RESULTS_FILE}")
metrics_frame


## 7. Preview Results

Review a sample of the exported results file before using it for charts or model diagnostics.


In [ ]:
results_frame.head(10)
